# TurnWave — text end-of-turn model (Colab)

**Before running anything:** Runtime -> Change runtime type -> **T4 GPU** -> Save.

Then Runtime -> Run all. Full training is ~30-60 min on a T4.

In [ ]:
# 1. Confirm we actually got a GPU. If this fails, fix the runtime type above.
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# 2. Clone and install. Colab ships its own CUDA torch, so --no-deps keeps it
# (our lockfile pins a CPU build). The assert makes a failed install loud —
# silently continuing here just produces confusing errors in later cells.
import subprocess, sys, os

REPO_URL = "https://github.com/Nikhils-G/turnwave.git"
if not os.path.isdir("/content/turnwave"):
    !git clone -q $REPO_URL /content/turnwave
%cd /content/turnwave
!git pull -q

install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".",
     "--no-deps", "sentencepiece", "datasets"]
)
assert install.returncode == 0, "install failed — read the error above before continuing"
print("install OK")

In [ ]:
# 3. Build the dataset (~170k pairs) and train the BPE tokenizer. ~2-3 min.
!python scripts/build_text_dataset.py --out data/text
!python -m turnwave.tokenizer data/text/corpus.txt checkpoints/tokenizer

In [ ]:
# 4. Train. First line must read device=cuda — if it says cpu, stop and fix cell 1.
!python -m turnwave.train \
    --train data/text/train.jsonl --val data/text/validation.jsonl \
    --tokenizer checkpoints/tokenizer/spm.model --out checkpoints/text_eot \
    --steps 6000 --batch-size 256 --num-workers 2

In [ ]:
# 5. The table that matters: our model vs the majority-class and cue-word baselines.
!python -m turnwave.evaluate \
    --ckpt checkpoints/text_eot/best.pt \
    --tokenizer checkpoints/tokenizer/spm.model \
    --data data/text/test.jsonl --device cuda

In [ ]:
# 6. Save the weights and the training log locally before the session expires.
from google.colab import files
files.download("checkpoints/text_eot/best.pt")
files.download("checkpoints/text_eot/log.csv")